# Advanced Modeling & Hyperparameter Tuning - XGBoost

This notebook covers training an advanced XGBoost model, tuning its hyperparameters using `RandomizedSearchCV` to optimize ROC-AUC, and persisting the best model.

In [ ]:
import os
import joblib
import pandas as pd
import numpy as np
from pathlib import Path
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV
from src.config import (
    PROCESSED_TRAIN_PATH,
    PROCESSED_VAL_PATH,
    TARGET_COLUMN,
    RANDOM_STATE,
    MODEL_DIR,
    setup_logging
)
from src.feature_engineering import build_feature_pipeline
from src.evaluate import evaluate_predictions

setup_logging()

## 1. Load Data & Transform Features

In [ ]:
train_df = pd.read_csv(PROCESSED_TRAIN_PATH, index_col=0)
val_df = pd.read_csv(PROCESSED_VAL_PATH, index_col=0)

X_train = train_df.drop(columns=[TARGET_COLUMN])
y_train = train_df[TARGET_COLUMN]
X_val = val_df.drop(columns=[TARGET_COLUMN])
y_val = val_df[TARGET_COLUMN]

pipeline = build_feature_pipeline()
X_train_trans = pipeline.fit_transform(X_train, y_train)
X_val_trans = pipeline.transform(X_val)

## 2. Hyperparameter Tuning using RandomizedSearchCV

In [ ]:
param_dist = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 4, 5, 6],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
    'min_child_weight': [1, 3, 5],
    'scale_pos_weight': [1, 5, 10, 13.9]
}

xgb = XGBClassifier(random_state=RANDOM_STATE, eval_metric='logloss')

search = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=param_dist,
    n_iter=10,
    scoring='roc_auc',
    cv=3,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1
)

search.fit(X_train_trans, y_train)

## 3. Best Model Performance & Parameters

In [ ]:
best_model = search.best_estimator_
print(f"Best Parameters: {search.best_params_}")
print(f"Best CV ROC-AUC: {search.best_score_:.4f}")

y_pred = best_model.predict(X_val_trans)
y_prob = best_model.predict_proba(X_val_trans)[:, 1]

plot_dir = Path("../notebooks/plots")
evaluate_predictions(y_val, y_pred, y_prob, "XGBoost_Tuned", plot_dir)

## 4. Save Artifacts

In [ ]:
MODEL_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(best_model, MODEL_DIR / "model.pkl")
joblib.dump(pipeline, MODEL_DIR / "preprocessor.pkl")
print("Best model and fitted preprocessor saved successfully.")